<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 - Notebook: Grafos de Logística de Insumos, Processo e Produtos Acabados
## Célula de Manufatura Flexível (FMS) Integrada à Planta Fabril

Neste notebook implementamos a modelagem em **Redes Multicamada (*Multilayer Networks*)** para integrar a Célula de Manufatura Flexível ao ecossistema logístico da fábrica:

1. **$G_M$ — Grafo de Fluxo de Materiais (DAG):** Mapeia o percurso unidirecional dos insumos desde a portaria rodoviária, passando pelo descarregamento na doca do Galpão A, segregação em boxes de tarugos e pigmentos, alimentação no `SILO-100`, inspeção e triagem na FMS, loteamento nas caixas `CX-301..303` e `DESC-304`, paletização automatizada, endereçamento WMS no Galpão B e expedição nas docas.
2. **$G_V$ — Grafo de Circulação de Veículos:** Mapeia o anel viário do pátio industrial, permitindo a circulação de caminhões de fornecedores, empilhadeiras pesadas e veículos de expedição com sentido dirigido e vias de desvio rápido.

O motor computacional executa:
* **Algoritmo de Kahn (1962):** Verificação formal de aciclicidade ($O(|V|+|E|)$), provando que $G_M$ é um DAG estrito e detectando os ciclos viários estruturais em $G_V$;
* **Algoritmo de Dijkstra com Min-Heap:** Cálculo das rotas de menor distância física com rastreamento detalhado de cada etapa industrial;
* **Relatórios de Auditoria SCADA / WMS:** Tabelas alinhadas em formato ASCII para integração com sistemas supervisórios industriais.

In [1]:
import heapq
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionários em tabela ASCII pura alinhada."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str

class GrafoLogistico:
    """
    Motor Computacional de Logística Fabril e Roteamento Multicamada.
    Suporta grafos de fluxo de materiais (DAG) e de circulação viária de veículos.
    """
    def __init__(self, nome: str = "GrafoLogistico"):
        self.nome = nome
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        """Adiciona uma rota orientada e ponderada com descrição operacional."""
        if peso < 0:
            raise ValueError("Pesos logísticos devem ser estritamente não-negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def vertices(self) -> List[str]:
        """Retorna a lista de todos os nós registrados na malha."""
        return list(self._adjacencias.keys())

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str], List[str]]:
        """
        Executa o Algoritmo de Dijkstra com Min-Heap binário para determinar
        a rota de menor custo métrico e extrair as descrições de cada etapa.
        """
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError(f"Origem ({origem}) ou destino ({destino}) ausente na malha logística.")

        dist = {v: float('inf') for v in self._adjacencias}
        pred = {v: None for v in self._adjacencias}
        desc_aresta = {v: "" for v in self._adjacencias}
        dist[origem] = 0.0
        fila = [(0.0, origem)]

        while fila:
            d_u, u = heapq.heappop(fila)
            if d_u > dist[u]:
                continue
            if u == destino:
                break
            for r in self._adjacencias[u]:
                alt = d_u + r.peso
                if alt < dist[r.destino]:
                    dist[r.destino] = alt
                    pred[r.destino] = u
                    desc_aresta[r.destino] = r.descricao
                    heapq.heappush(fila, (alt, r.destino))

        if dist[destino] == float('inf'):
            raise ValueError(f"Não existe rota viável conectando '{origem}' a '{destino}'.")

        caminho = []
        descricoes = []
        curr = destino
        while curr is not None:
            caminho.append(curr)
            if pred[curr] is not None:
                descricoes.append(desc_aresta[curr])
            curr = pred[curr]
        caminho.reverse()
        descricoes.reverse()
        return round(dist[destino], 1), caminho, descricoes

    def ordenacao_topologica_kahn(self) -> Tuple[bool, List[str]]:
        """
        Executa o Algoritmo de Kahn (1962) para verificar se a rede é um Grafo Acíclico Dirigido (DAG)
        e extrair uma ordenação linear compatível com as regras de precedência da manufatura.
        """
        grau_entrada = {v: 0 for v in self._adjacencias}
        for u in self._adjacencias:
            for r in self._adjacencias[u]:
                grau_entrada[r.destino] += 1

        fila = [v for v, g in grau_entrada.items() if g == 0]
        ordenados = []

        while fila:
            u = fila.pop(0)
            ordenados.append(u)
            for r in self._adjacencias[u]:
                grau_entrada[r.destino] -= 1
                if grau_entrada[r.destino] == 0:
                    fila.append(r.destino)

        eh_dag = (len(ordenados) == len(self._adjacencias))
        return eh_dag, ordenados

    def auditar_detalhes_rota(self, origem: str, destino: str) -> List[Dict[str, Any]]:
        """Gera relatório operacional tabulado em formato ASCII alinhado para o SCADA."""
        custo_total, caminho, descricoes = self.menor_caminho(origem, destino)
        tabela = []
        acumulado = 0.0
        for i in range(len(caminho) - 1):
            u = caminho[i]
            v = caminho[i+1]
            peso = next(r.peso for r in self._adjacencias[u] if r.destino == v)
            acumulado += peso
            tabela.append({
                "Etapa": f"{i+1}/{len(caminho)-1}",
                "Origem": u,
                "Destino": v,
                "Distância (m)": f"{peso:.1f}",
                "Acumulado (m)": f"{acumulado:.1f}",
                "Operação Logística": descricoes[i]
            })
        return tabela

# --- 1. MODELO GM: FLUXO FÍSICO DE MATERIAIS (DAG) ---
gm = GrafoLogistico("Fluxo de Materiais FMS")
rotas_gm = [
    # 1. Recebimento e Almoxarifado de Entrada (Galpão A)
    ("Portaria", "Balança de Entrada", 25.0, "Liberação e pesagem rodoviária do caminhão"),
    ("Balança de Entrada", "Doca de Descarga (Galpão A)", 45.0, "Direcionamento à descarga de matérias-primas"),
    ("Doca de Descarga (Galpão A)", "Box: Tarugos Grandes (Vermelho/Verde)", 15.0, "Armazenagem de tarugos cilíndricos grandes"),
    ("Doca de Descarga (Galpão A)", "Box: Tarugos Pequenos (Azul)", 18.0, "Armazenagem de tarugos cilíndricos pequenos"),
    ("Doca de Descarga (Galpão A)", "Box: Pigmentos Industriais (RGB)", 20.0, "Armazenagem de pigmentos industriais de acabamento"),
    ("Doca de Descarga (Galpão A)", "Box: Embalagens e Caixas de Lote", 22.0, "Armazenagem de insumos de empacotamento"),

    # 2. Transferência para Célula FMS
    ("Box: Tarugos Grandes (Vermelho/Verde)", "Posto de Preparação e Carga", 25.0, "Despaletização e alimentação"),
    ("Box: Tarugos Pequenos (Azul)", "Posto de Preparação e Carga", 22.0, "Despaletização e alimentação"),
    ("Box: Pigmentos Industriais (RGB)", "Estação de Dosagem de Pigmento", 24.0, "Alimentação de tintas e corantes"),
    ("Posto de Preparação e Carga", "SILO-100: Silo Infeed", 18.0, "Abastecimento do silo alimentador da esteira"),
    ("Estação de Dosagem de Pigmento", "INSP-200: Túnel Óptico", 20.0, "Transferência para banho/acabamento óptico"),

    # 3. Processamento e Triagem na Célula de Manufatura (Setores 100, 200 e 300)
    ("SILO-100: Silo Infeed", "INSP-200: Túnel Óptico", 12.0, "Transporte inicial e inspeção dimensional/cor"),
    ("INSP-200: Túnel Óptico", "DIV-201: Ponto de Desvio 1", 10.0, "Encaminhamento à triagem automática"),
    ("DIV-201: Ponto de Desvio 1", "CX-301: Lote Grandes Vermelhas", 5.0, "Desvio pneumático XV-201 ativado"),
    ("DIV-201: Ponto de Desvio 1", "DIV-202: Ponto de Desvio 2", 8.0, "Esteira principal M-101"),
    ("DIV-202: Ponto de Desvio 2", "CX-302: Lote Grandes Verdes", 5.0, "Desvio pneumático XV-202 ativado"),
    ("DIV-202: Ponto de Desvio 2", "DIV-203: Ponto de Desvio 3", 8.0, "Esteira principal M-101"),
    ("DIV-203: Ponto de Desvio 3", "CX-303: Lote Pequenas Azuis", 5.0, "Desvio pneumático XV-203 ativado"),
    ("DIV-203: Ponto de Desvio 3", "DESC-304: Calha de Descarte", 8.0, "Descarte de refugo fora de padrão M-101"),

    # 4. Loteamento, Paletização e Armazém WMS (Galpão B)
    ("CX-301: Lote Grandes Vermelhas", "Paletizadora Automática", 20.0, "Formação de lote (10 peças empacotadas)"),
    ("CX-302: Lote Grandes Verdes", "Paletizadora Automática", 22.0, "Formação de lote (10 peças empacotadas)"),
    ("CX-303: Lote Pequenas Azuis", "Paletizadora Automática", 25.0, "Formação de lote (10 peças empacotadas)"),
    ("Box: Embalagens e Caixas de Lote", "Paletizadora Automática", 30.0, "Alimentação contínua de paletes e caixas"),

    ("Paletizadora Automática", "Estoque: Lote Grandes Vermelhas", 20.0, "Endereçamento WMS em porta-paletes"),
    ("Paletizadora Automática", "Estoque: Lote Grandes Verdes", 22.0, "Endereçamento WMS em porta-paletes"),
    ("Paletizadora Automática", "Estoque: Lote Pequenas Azuis", 25.0, "Endereçamento WMS em porta-paletes"),

    ("DESC-304: Calha de Descarte", "Área de Sucata e Reciclagem", 18.0, "Coleta e triagem de refugos para reutilização"),

    # 5. Expedição Final
    ("Estoque: Lote Grandes Vermelhas", "Docas de Expedição", 25.0, "Separação e carregamento em caminhão"),
    ("Estoque: Lote Grandes Verdes", "Docas de Expedição", 22.0, "Separação e carregamento em caminhão"),
    ("Estoque: Lote Pequenas Azuis", "Docas de Expedição", 28.0, "Separação e carregamento em caminhão"),
    ("Área de Sucata e Reciclagem", "Doca de Sucata e Resíduos", 15.0, "Despacho para usina de reciclagem")
]
for u, v, w, d in rotas_gm:
    gm.adicionar_rota(u, v, w, d)

# --- 2. MODELO GV: CIRCULAÇÃO INTERNA DE VEÍCULOS NO ANEL FABRIL ---
gv = GrafoLogistico("Circulação de Veículos")
rotas_gv = [
    ("Portaria", "Balança de Entrada", 25.0, "Entrada rodoviária de veículos"),
    ("Balança de Entrada", "Pátio de Carga/Descarga (Galpão A)", 40.0, "Acesso à descarga de matéria-prima"),
    ("Pátio de Carga/Descarga (Galpão A)", "Pavilhão FMS (Acesso de Serviço)", 50.0, "Circulação interna de veículos"),
    ("Pavilhão FMS (Acesso de Serviço)", "Pátio de Expedição (Galpão B)", 45.0, "Circulação interna de carga"),
    ("Pátio de Expedição (Galpão B)", "Docas de Expedição", 30.0, "Posicionamento em baia de carregamento"),
    ("Docas de Expedição", "Balança de Saída", 55.0, "Deslocamento para pesagem de saída"),
    ("Balança de Saída", "Portaria", 30.0, "Liberação final na portaria"),
    # Rota de contingência / desvio rápido
    ("Pátio de Carga/Descarga (Galpão A)", "Balança de Saída", 70.0, "Desvio rápido de caminhões vazios descarregados")
]
for u, v, w, d in rotas_gv:
    gv.adicionar_rota(u, v, w, d)

# --- 3. EXECUÇÕES E DEMONSTRAÇÕES ---

# 1. Auditoria de Aciclicidade via Algoritmo de Kahn
print("=== 1. AUDITORIA ESTRUTURAL DE ACICLICIDADE (ALGORITMO DE KAHN) ===")
eh_dag_gm, ord_gm = gm.ordenacao_topologica_kahn()
print(f"Grafo GM (Materiais): {'É um DAG válido!' if eh_dag_gm else 'Contém ciclos!'} (Total de vértices: {len(gm.vertices())})")
print("Sequência Topológica Parcial: " + " -> ".join(ord_gm[:6]) + " ... -> " + ord_gm[-1])

eh_dag_gv, ord_gv = gv.ordenacao_topologica_kahn()
print(f"Grafo GV (Veículos):   {'É um DAG válido!' if eh_dag_gv else 'NÃO é DAG (Contém Ciclos Estruturais)'} (Total de vértices: {len(gv.vertices())})")
print("Diagnóstico: O anel viário GV contém ciclo fechado (Portaria -> Balança IN -> ... -> Docas -> Balança OUT -> Portaria).")

# 2. Menor Caminho Portaria -> Docas de Expedição
print("\n=== 2. MENOR CAMINHO DE ENTRADA ATÉ A EXPEDIÇÃO EM GM ===")
custo_p, caminho_p, _ = gm.menor_caminho("Portaria", "Docas de Expedição")
print("Rota:", " -> ".join(caminho_p))
print(f"Distância Métrica Total: {custo_p:.1f} m\n")
print("Detalhamento Etapa por Etapa:")
tabela_portaria = gm.auditar_detalhes_rota("Portaria", "Docas de Expedição")
print(formatar_tabela(tabela_portaria))

# 3. Auditoria de Matérias-Primas até as Docas
print("\n=== 3. AUDITORIA DE RASTREAMENTO DAS MATÉRIAS-PRIMAS ATÉ AS DOCAS ===")
insumos = [
    "Box: Tarugos Grandes (Vermelho/Verde)",
    "Box: Tarugos Pequenos (Azul)",
    "Box: Pigmentos Industriais (RGB)",
    "Box: Embalagens e Caixas de Lote"
]
rel_insumos = []
for ins in insumos:
    c, cam, _ = gm.menor_caminho(ins, "Docas de Expedição")
    resumo_cam = cam[0].split(':')[0] + " -> " + " -> ".join([n.split(':')[0] for n in cam[1:-1]]) + " -> " + cam[-1]
    if len(resumo_cam) > 65:
        resumo_cam = resumo_cam[:62] + "..."
    rel_insumos.append({
        "Insumo de Origem": ins,
        "Distância (m)": f"{c:.1f}",
        "Saltos": len(cam) - 1,
        "Rota Ótima (Dijkstra)": resumo_cam
    })
print(formatar_tabela(rel_insumos))

# 4. Roteamento do Processo FMS: SILO até os Estoques Segregados
print("\n=== 4. ROTEAMENTO DO PROCESSO FMS: SILO ATÉ OS ESTOQUES FINAIS ===")
destinos_processo = [
    "Estoque: Lote Grandes Vermelhas",
    "Estoque: Lote Grandes Verdes",
    "Estoque: Lote Pequenas Azuis",
    "Área de Sucata e Reciclagem"
]
rel_processo = []
for dst in destinos_processo:
    c, cam, _ = gm.menor_caminho("SILO-100: Silo Infeed", dst)
    rel_processo.append({
        "Destino / Estoque WMS": dst,
        "Distância (m)": f"{c:.1f}",
        "Setor FMS": "Setor 300",
        "Rota Específica de Fabricação": " -> ".join([n.split(':')[0] for n in cam])
    })
print(formatar_tabela(rel_processo))

# 5. Circulação de Veículos no Anel Fabril (GV)
print("\n=== 5. CIRCULAÇÃO VIÁRIA DE VEÍCULOS NO ANEL FABRIL (GV) ===")
cV1, pV1, _ = gv.menor_caminho("Portaria", "Docas de Expedição")
cV2, pV2, _ = gv.menor_caminho("Docas de Expedição", "Portaria")
cV3, pV3, _ = gv.menor_caminho("Pátio de Carga/Descarga (Galpão A)", "Portaria")
rel_veiculos = [
    {"Missão Viária": "Entrada de Veículo", "Distância (m)": f"{cV1:.1f}", "Rota Viária Selecionada": " -> ".join(pV1)},
    {"Missão Viária": "Saída de Veículo", "Distância (m)": f"{cV2:.1f}", "Rota Viária Selecionada": " -> ".join(pV2)},
    {"Missão Viária": "Desvio Rápido Vago", "Distância (m)": f"{cV3:.1f}", "Rota Viária Selecionada": " -> ".join(pV3)}
]
print(formatar_tabela(rel_veiculos))

# --- 6. ASSERÇÕES DETERMINÍSTICAS DE ENGENHARIA ---
# Validação Estrutural de Grafos
assert eh_dag_gm is True, "O Grafo de Materiais GM deve ser estritamente um DAG acíclico."
assert eh_dag_gv is False, "O Grafo de Veículos GV deve conter ciclos devido ao retorno à portaria."
assert len(gm.vertices()) == 25, "GM deve possuir 25 vértices correspondentes à planta completa."
assert len(gv.vertices()) == 7, "GV deve possuir 7 postos de circulação viária."

# Validação dos Menores Caminhos em GM
assert custo_p == 166.0
assert "Box: Embalagens e Caixas de Lote" in caminho_p
assert "Paletizadora Automática" in caminho_p

# Validação do trajeto de produção do Silo
c_silo_verm, p_silo_verm, _ = gm.menor_caminho("SILO-100: Silo Infeed", "Estoque: Lote Grandes Vermelhas")
assert c_silo_verm == 67.0
assert "CX-301: Lote Grandes Vermelhas" in p_silo_verm

c_silo_sucata, p_silo_sucata, _ = gm.menor_caminho("SILO-100: Silo Infeed", "Área de Sucata e Reciclagem")
assert c_silo_sucata == 64.0
assert "DESC-304: Calha de Descarte" in p_silo_sucata

# Validação de Circulação Viária em GV
assert cV1 == 190.0
assert cV2 == 85.0
assert pV2 == ["Docas de Expedição", "Balança de Saída", "Portaria"]
assert cV3 == 100.0
assert pV3 == ["Pátio de Carga/Descarga (Galpão A)", "Balança de Saída", "Portaria"]

print("\n[OK] Modelos Logísticos Multicamada da FMS validados com sucesso!")

=== 1. AUDITORIA ESTRUTURAL DE ACICLICIDADE (ALGORITMO DE KAHN) ===
Grafo GM (Materiais): É um DAG válido! (Total de vértices: 25)
Sequência Topológica Parcial: Portaria -> Balança de Entrada -> Doca de Descarga (Galpão A) -> Box: Tarugos Grandes (Vermelho/Verde) -> Box: Tarugos Pequenos (Azul) -> Box: Pigmentos Industriais (RGB) ... -> Docas de Expedição
Grafo GV (Veículos):   NÃO é DAG (Contém Ciclos Estruturais) (Total de vértices: 7)
Diagnóstico: O anel viário GV contém ciclo fechado (Portaria -> Balança IN -> ... -> Docas -> Balança OUT -> Portaria).

=== 2. MENOR CAMINHO DE ENTRADA ATÉ A EXPEDIÇÃO EM GM ===
Rota: Portaria -> Balança de Entrada -> Doca de Descarga (Galpão A) -> Box: Embalagens e Caixas de Lote -> Paletizadora Automática -> Estoque: Lote Grandes Verdes -> Docas de Expedição
Distância Métrica Total: 166.0 m

Detalhamento Etapa por Etapa:
Etapa | Origem                       | Destino                      | Distância (m) | Acumulado (m) | Operação Logística          